# Singular Value Decomposition (SVD)

_Linear Algebra_

**Factor ANY matrix into rotate, stretch, rotate. Keep the big stretches, throw the rest away.**

The SVD (Singular Value Decomposition) factors any matrix (square or not) as $A = U\Sigma V^\top$.

     $V^\top$ rotates, $\Sigma$ stretches each axis by a non-negative singular value, and $U$ rotates again. The singular values $\sigma_1 \ge \sigma_2 \ge \dots$ rank the directions from most important to least.

---

This notebook is a practice scaffold for the **Singular Value Decomposition (SVD)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

### Step 1 — Factor the matrix into U, Σ, Vᵀ

We build a small random 4×3 matrix `A` and hand it to `np.linalg.svd`. The decomposition returns three pieces: `U` (left rotation), `s` (the singular values, a 1-D array sorted largest-first), and `Vt` (the right rotation, already transposed). `full_matrices=False` gives the economy-size factors that are just big enough to rebuild `A`.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# A small random matrix to factor (4 rows, 3 columns).
A = rng.standard_normal((4, 3))

# SVD returns U, the singular values s (sorted high to low), and V transposed.
U, s, Vt = np.linalg.svd(A, full_matrices=False)
print("singular values:", np.round(s, 4))

### Step 2 — Rebuild A exactly from the factors

The singular values come back as a flat list, so we place them on the diagonal of a matrix with `np.diag(s)` before multiplying. Multiplying `U @ diag(s) @ Vt` reconstructs the original `A` exactly (up to floating-point rounding), which `np.allclose` confirms.

In [ ]:
# Put the singular values on a diagonal so the matrix product lines up.
S = np.diag(s)

# Multiply the three factors back together to recover A.
A_rebuilt = U @ S @ Vt
print("A == U S V^T:", np.allclose(A, A_rebuilt))

### Step 3 — Keep only the top singular value (rank-1 approximation)

The Eckart–Young theorem says the best rank-`k` approximation keeps the `k` largest singular values and drops the rest. Here we keep just the first one (`k = 1`), so `A1` collapses to rank 1. The error of that approximation, measured by the Frobenius norm, exactly equals the first singular value we threw away — `s[1]`.

In [ ]:
# Truncate to the top-1: the best rank-1 approximation (Eckart-Young).
k = 1
A1 = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
print("rank of A1:", np.linalg.matrix_rank(A1))

# The leftover error equals the first dropped singular value.
err = np.linalg.norm(A - A1)
print("rank-1 error =", round(err, 4), " (= next singular value", round(s[1], 4), ")")

## Visualize the data & results

_How fast do the singular values decay, and how does that bound the rank-k reconstruction error?_

### Step 1 — Compute the reconstruction error at every rank

We re-factor the same matrix, then sweep `k` from 0 (keep nothing) up to the full rank (keep everything). At each `k` we rebuild the rank-`k` approximation and record its Frobenius distance from `A`. The errors should shrink to zero as we add back more singular values.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

A = rng.standard_normal((4, 3))
U, s, Vt = np.linalg.svd(A, full_matrices=False)   # singular values decay

errors = []
for k in range(len(s) + 1):                         # rank-k reconstruction
    Ak = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    errors.append(np.linalg.norm(A - Ak))           # Frobenius error

### Step 2 — Plot the singular-value decay and the error curve

Left: the singular values themselves, which fall off from largest to smallest — the steeper the decay, the more a low-rank approximation captures. Right: the reconstruction error against how many singular values we keep, dropping to zero once we keep them all.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))

# Left — the singular values, sorted largest to smallest.
ax1.bar(['sigma 1', 'sigma 2', 'sigma 3'], s, color='#4ea1ff')
ax1.set_title('singular values (decaying)')

# Right — how the error falls as we keep more singular values.
ax2.plot(range(len(errors)), errors, marker='o', color='#7ee787')
ax2.set_xlabel('rank k kept')
ax2.set_ylabel('||A - A_k||')
ax2.set_title('reconstruction error vs rank')

plt.tight_layout()
plt.show()